# Lección 13.4 — Modelos de difusión: crear imágenes quitando ruido

> **Módulo 13 · Modelos generativos y LLMs** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 13.2 (autoencoders, en especial el que quita ruido), Lección 13.3 (GANs)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía la idea de la **difusión**: destruir datos con ruido y aprender a deshacerlo.
- Distinguir el **proceso hacia adelante** (añadir ruido, sin aprender nada) del **proceso inverso** (quitar ruido, lo que aprende la red).
- Entrenar un modelo de difusión de juguete que **predice el ruido** y usarlo para generar datos nuevos desde ruido puro.
- Predecir el efecto de sus perillas: **número de pasos T**, **calendario de ruido** y **pasos de muestreo**.
- Entender cómo se **guía** la generación (por ejemplo, con un texto en Stable Diffusion).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas, un dígito de MNIST y nuestros datos de juguete: puntos en forma de dos "lunas".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4)

# Un dígito de MNIST, con píxeles escalados a [-1, 1] (lo habitual en difusión)
mnist = datasets.MNIST(root="data", train=True, download=True)
digito = mnist.data[0].numpy() / 255 * 2 - 1

# Datos de juguete: 3,000 puntos en forma de dos lunas, y a qué luna pertenece cada uno
lunas_np, luna_de = make_moons(n_samples=3000, noise=0.04, random_state=0)
LUNAS = torch.tensor(lunas_np, dtype=torch.float32)
ETIQUETAS = torch.tensor(luna_de, dtype=torch.float32)

def evaluar(muestras, reales=lunas_np):
    """Precisión: % de puntos generados que caen sobre la forma real.
       Cobertura: % de la forma real que tiene algún punto generado cerca."""
    muestras = np.asarray(muestras)
    precision = (NearestNeighbors(n_neighbors=1).fit(reales).kneighbors(muestras)[0][:, 0] < 0.1).mean()
    cobertura = (NearestNeighbors(n_neighbors=1).fit(muestras).kneighbors(reales)[0][:, 0] < 0.1).mean()
    return precision, cobertura

plt.figure(figsize=(5, 3)); plt.scatter(*lunas_np.T, s=2); plt.title("Datos reales: dos lunas"); plt.show()
print("✅ Todo listo")

---
## 1. 📖 La intuición: una gota de tinta, en reversa

Deja caer una gota de tinta en un vaso de agua. Al principio ves una forma nítida; después se **difunde** poco a poco hasta que el agua queda de un gris uniforme. Ese proceso es fácil: ocurre solo.

```
 💧 forma nítida  ─►  🌫️ algo borrosa  ─►  🌫️🌫️ muy borrosa  ─►  ▒▒▒ gris uniforme
     (datos)              + ruido             + más ruido           (ruido puro)
```

Ahora imagina ver la película **al revés**: del agua gris surge, poco a poco, una forma nítida. En la física eso nunca pasa… pero una red neuronal **puede aprender a hacerlo**, si la entrenamos para **deshacer un poquito de ruido cada vez**.

**Un modelo de difusión** hace eso: aprende a quitar ruido. Para **crear** algo nuevo, empieza desde ruido puro y lo va limpiando paso a paso hasta que aparece una imagen. Es la técnica detrás de **Stable Diffusion, DALL·E, Midjourney** y muchos generadores de video.

### ✋ Ejercicio 1 — Calentamiento
En la Lección 13.2 entrenaste un **autoencoder que quita ruido**. ¿En qué se parece a lo que acabas de leer? ¿Qué le faltaba para poder **crear** imágenes desde cero?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Se parecen en lo esencial: los dos aprenden a **quitar ruido** (entrada sucia → salida limpia).

Lo que le faltaba al autoencoder: solo sabía limpiar **un nivel fijo** de ruido (0.4) y lo hacía **de un solo golpe**. Un modelo de difusión aprende a limpiar **todos los niveles**, desde "casi limpio" hasta "ruido puro", y lo hace **en muchos pasitos**. Así puede empezar desde ruido puro, donde no queda nada de imagen, e ir "esculpiendo" una.
</details>

---
## 2. 📖 El concepto: dos procesos

| Proceso | Qué hace | ¿Se aprende? | En la analogía |
|---|---|---|---|
| **Hacia adelante** (*forward*) | Añade ruido poco a poco durante **T** pasos hasta llegar a ruido puro | ❌ No: es una fórmula fija | La tinta se difunde |
| **Inverso** (*reverse*) | Quita un poco de ruido en cada paso, de *T* a 0 | ✅ Sí: lo hace la red | La película al revés |

**¿Qué aprende exactamente la red?** Le damos un dato ensuciado y le decimos **cuánto** ruido tiene (el paso *t*). Su tarea: **adivinar el ruido** que se le añadió. Si sabes qué ruido tiene una imagen, puedes restárselo.

```
 entrada: (dato con ruido, paso t)  ──►  [ RED ]  ──►  "creo que el ruido era este"
 pérdida: error cuadrático entre el ruido adivinado y el ruido real   (¡aprendizaje supervisado normal!)
```

La gran ventaja frente a las GANs (13.3): **no hay ningún juego entre dos redes**. Es un problema de regresión estable, como los del Módulo 5, solo que con millones de ejemplos fabricados al vuelo.

### ✋ Ejercicio 2 — ¿De dónde salen los ejemplos?
Para entrenar la red necesitamos pares (entrada, respuesta correcta). Si solo tienes las 3,000 lunas limpias:
1. ¿Cómo fabricas un par de entrenamiento?
2. ¿Cuántos pares distintos puedes fabricar?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Tomas un punto limpio, eliges un paso *t* al azar, sorteas un ruido y se lo añades con la fórmula del proceso hacia adelante. **Entrada** = (punto ensuciado, *t*); **respuesta correcta** = el ruido que sorteaste. ¡Tú lo sabes porque tú lo añadiste!
2. **Prácticamente infinitos:** cada vez que sorteas otro ruido u otro *t* obtienes un par nuevo. Por eso estos modelos aprovechan tan bien los datos.
</details>

---
## 3. 📖 Cómo funciona por dentro

### 3.1 Primero "a mano": el proceso hacia adelante
La fórmula para ensuciar un dato hasta el paso *t* de un solo golpe es una **mezcla**:

```
dato_con_ruido = √(señal) × dato_limpio  +  √(1 − señal) × ruido
```

donde `señal` va de 1 (paso 0, todo dato) a casi 0 (paso T, todo ruido). La lista de valores de `señal` para cada paso se llama **calendario de ruido** (*noise schedule*). Usaremos el calendario **coseno**, muy común. Vamos a ensuciar un dígito con numpy.

In [ ]:
def calendario(T=100, tipo="coseno"):
    """Devuelve, para cada paso t = 1..T, qué fracción de 'señal' queda (de ≈1 a ≈0)."""
    s = np.arange(1, T + 1) / T
    if tipo == "coseno":
        senal = np.cos((s + 0.008) / 1.008 * np.pi / 2) ** 2 / np.cos(0.008 / 1.008 * np.pi / 2) ** 2
    else:  # "lineal": se añade la misma cantidad de ruido en cada paso
        senal = np.cumprod(1 - np.linspace(1e-4, 0.1, T))
    return np.clip(senal, 1e-4, 1)

T = 100
SENAL = calendario(T)
rng = np.random.default_rng(0)
ruido = rng.standard_normal(digito.shape)

pasos_a_ver = [0, 10, 25, 50, 75, 99]          # ✏️ cambia los pasos
fig, axes = plt.subplots(1, len(pasos_a_ver), figsize=(12, 2.4))
for ax, t in zip(axes, pasos_a_ver):
    sucio = np.sqrt(SENAL[t]) * digito + np.sqrt(1 - SENAL[t]) * ruido     # ← la fórmula
    ax.imshow(sucio, cmap="gray", vmin=-1.5, vmax=1.5); ax.axis("off")
    ax.set_title(f"paso {t + 1}\nseñal = {SENAL[t]:.2f}", fontsize=9)
plt.suptitle("Proceso hacia adelante: el dígito se disuelve en ruido"); plt.show()

### 🔮 Ejercicio 3 — ¿Hasta cuándo se reconoce?
Antes de mirar con detalle: ¿en qué paso (de 100) crees que el dígito deja de reconocerse? ¿A la mitad? ¿Antes? ¿Después?

Luego mira la figura y ✏️ prueba otros pasos en `pasos_a_ver` (por ejemplo `[30, 40, 50, 60, 70]`).

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

En la figura, el 5 se reconoce sin problema en los pasos 1, 11 y 26, todavía se adivina en el **51** (señal ≈ 0.5) y en el **76** ya no se ve (señal ≈ 0.13). Con más detalle, deja de reconocerse hacia el **paso 60–70**.

Esto importa para generar: **en los primeros pasos del proceso inverso (cerca de T) se decide la forma general** (¿qué dígito?, ¿dónde está?), y **en los últimos (cerca de 0) se pulen los detalles**. Un buen calendario reparte bien el ruido entre ambas fases.
</details>

▶️ Lo mismo con nuestras lunas: cada punto se mueve un poco más hacia el azar en cada paso.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, t in zip(axes, [0, 30, 60, 99]):
    ruido_2d = rng.standard_normal(lunas_np.shape)
    sucias = np.sqrt(SENAL[t]) * lunas_np + np.sqrt(1 - SENAL[t]) * ruido_2d
    ax.scatter(*sucias.T, s=1); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_title(f"paso {t + 1} · señal = {SENAL[t]:.2f}", fontsize=10)
plt.suptitle("Las lunas se 'difunden' hasta ser una nube de ruido"); plt.show()

### 3.2 Entrenar la red que adivina el ruido (PyTorch)
La red es un perceptrón pequeño (Módulo 9). Recibe **3 números**: las dos coordenadas del punto sucio y el paso *t* (escalado a 0–1). Devuelve **2 números**: su apuesta del ruido. Más adelante usaremos la opción `etiquetas` para **guiar** la generación.

In [ ]:
def entrenar_difusion(datos=LUNAS, T=100, pasos=4000, tipo="coseno", etiquetas=None, semilla=0):
    torch.manual_seed(semilla)
    senal = torch.tensor(calendario(T, tipo), dtype=torch.float32, device=device)
    extra = 0 if etiquetas is None else 1                 # con guía, la red también recibe la etiqueta
    red = nn.Sequential(nn.Linear(3 + extra, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                        nn.Linear(128, 2)).to(device)
    optimizador = torch.optim.Adam(red.parameters(), lr=1e-3)
    datos = datos.to(device)
    perdidas = []
    for paso in range(pasos):
        i = torch.randint(0, len(datos), (256,), device=device)
        limpio = datos[i]
        t = torch.randint(0, T, (256,), device=device)                  # un paso al azar para cada punto
        ruido = torch.randn_like(limpio)                                # el ruido que sorteamos
        s = senal[t].unsqueeze(1)
        sucio = s.sqrt() * limpio + (1 - s).sqrt() * ruido              # proceso hacia adelante
        entrada = [sucio, (t / T).unsqueeze(1)]
        if etiquetas is not None:
            entrada.append(etiquetas.to(device)[i].unsqueeze(1))
        perdida = ((red(torch.cat(entrada, dim=1)) - ruido) ** 2).mean()   # ¿adivinó el ruido?
        optimizador.zero_grad(); perdida.backward(); optimizador.step()
        perdidas.append(perdida.item())
    return {"red": red, "senal": senal, "T": T, "perdidas": perdidas, "guiado": etiquetas is not None}

modelo = entrenar_difusion()
plt.plot(np.convolve(modelo["perdidas"], np.ones(100) / 100, mode="valid"))
plt.xlabel("paso de entrenamiento"); plt.ylabel("error al adivinar el ruido"); plt.grid(alpha=.3)
plt.title("Curva de aprendizaje del modelo de difusión"); plt.show()
print(f"Pérdida media en los primeros 100 pasos: {np.mean(modelo['perdidas'][:100]):.2f}")
print(f"Pérdida media en los últimos 500 pasos : {np.mean(modelo['perdidas'][-500:]):.2f}")

### 🔮 Ejercicio 4 — Lee la curva
1. Compárala con las curvas de la GAN (Lección 13.3). ¿Qué diferencia ves?
2. La pérdida no llega a 0. ¿Por qué crees que es **imposible** adivinar el ruido perfectamente? (Pista: un mismo punto sucio, ¿de cuántos puntos limpios distintos pudo salir?)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Esta curva **baja** de forma tranquila (de ≈ 0.44 a ≈ 0.33) y se estabiliza, como en cualquier problema supervisado (Lección 3.7). No hay dos rivales tirando en direcciones opuestas: por eso entrenar difusión es mucho más **estable** que entrenar una GAN, y la pérdida sí sirve para vigilar el entrenamiento.
2. Porque hay **ambigüedad**: un mismo punto sucio pudo salir de muchos puntos limpios distintos con ruidos distintos. Por ejemplo, un punto que está un poco fuera de la luna, ¿es un punto limpio del borde con poco ruido o uno del centro con más ruido? La red solo puede apostar por el **promedio** de las posibilidades, y siempre queda un error. Eso no es un problema: para generar basta con apuntar en la dirección correcta paso a paso.
</details>

### 3.3 Generar: de ruido puro a lunas
Para crear puntos nuevos, empezamos con **ruido puro** y repetimos, del paso *T* al 1:
1. La red **adivina el ruido** del punto actual.
2. Con esa apuesta estimamos cómo sería el punto **limpio**.
3. Damos un paso hacia él, dejando un poco **menos** de ruido que antes.

(Esta forma de recorrer el camino se llama **DDIM**; permite usar menos pasos que *T*, lo que aprovecharemos enseguida.)

In [ ]:
@torch.no_grad()
def generar(modelo, n=2000, pasos=None, etiqueta=None, semilla=1, fotos_en=()):
    red, senal, T = modelo["red"], modelo["senal"], modelo["T"]
    torch.manual_seed(semilla)
    recorrido = torch.linspace(T - 1, 0, pasos or T).long()      # de T hacia 0 (quizá saltando pasos)
    x = torch.randn(n, 2, device=device)                         # ruido puro
    fotos = {}
    for k, t in enumerate(recorrido):
        entrada = [x, torch.full((n, 1), t.item() / T, device=device)]
        if modelo["guiado"]:
            entrada.append(torch.full((n, 1), float(etiqueta), device=device))
        ruido_estimado = red(torch.cat(entrada, dim=1))                                 # 1. adivinar el ruido
        limpio_estimado = (x - (1 - senal[t]).sqrt() * ruido_estimado) / senal[t].sqrt()  # 2. estimar el limpio
        senal_siguiente = senal[recorrido[k + 1]] if k + 1 < len(recorrido) else torch.tensor(1.0)
        x = senal_siguiente.sqrt() * limpio_estimado + (1 - senal_siguiente).sqrt() * ruido_estimado  # 3. paso
        if k + 1 in fotos_en:
            fotos[k + 1] = x.cpu().numpy()
    return (x.cpu().numpy(), fotos) if fotos_en else x.cpu().numpy()

print("✅ Función lista")

### 🔮 Ejercicio 5 — La película al revés
Generamos 2,000 puntos con los 100 pasos y tomamos fotos en los pasos 1, 25, 50, 75 y 100 del camino de regreso. ¿Cómo crees que se verá la foto de la **mitad** del camino (paso 50)?

✍️ *Mi predicción:*

In [ ]:
FOTOS = (1, 25, 50, 75, 100)
nuevos, fotos = generar(modelo, fotos_en=FOTOS)
fig, axes = plt.subplots(1, len(FOTOS), figsize=(16, 3.2))
for ax, k in zip(axes, FOTOS):
    ax.scatter(*fotos[k].T, s=1, c="crimson"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_title(f"paso {k} de 100 del regreso", fontsize=10)
plt.suptitle("Proceso inverso: el ruido se convierte en lunas"); plt.show()
precision, cobertura = evaluar(nuevos)
print(f"Precisión: {precision:.0%} de los puntos generados caen sobre las lunas")
print(f"Cobertura: {cobertura:.0%} de las lunas reales tienen un punto generado cerca")

<details><summary><b>👀 Ver respuesta</b></summary>

En el paso 50 del regreso todavía es una **nube difusa**, pero ya no es redonda: se ha inclinado y estirado en la dirección general de las lunas. En el paso 75 se adivina una forma de "S" gruesa, y solo en los **últimos pasos** se afilan las dos lunas. Es lo que viste en el Ejercicio 3: **primero se decide la forma gruesa, al final los detalles**.

El resultado final: ≈ **88 %** de precisión y ≈ **99 %** de cobertura. Quedan algunos puntos en el "puente" entre las dos lunas, pero cubre las dos lunas completas (sin colapso de modos, a diferencia de las GANs).
</details>

---
## 4. 📖 ¿Qué pasa si… genero con menos pasos?

Generar con 100 pasos significa llamar a la red **100 veces**. Una GAN (13.3) genera con **una sola** llamada. En imágenes reales eso marca la diferencia entre 0.05 s y 5 s por imagen. ¿Podemos ahorrar pasos?

### 🔮 Ejercicio 6 — Predice
Con el **mismo modelo** ya entrenado, generamos usando 1, 3, 10 y 100 pasos de muestreo.
1. ¿Qué crees que sale con **1 solo paso**?
2. ¿A partir de cuántos pasos crees que la calidad ya casi no mejora?

✍️ *Mis predicciones:*

In [ ]:
PASOS_MUESTREO = [1, 3, 10, 100]          # ✏️ prueba también 5 o 20
fig, axes = plt.subplots(1, len(PASOS_MUESTREO), figsize=(16, 3.2))
for ax, p in zip(axes, PASOS_MUESTREO):
    muestras = generar(modelo, pasos=p)
    precision, cobertura = evaluar(muestras)
    ax.scatter(*lunas_np.T, s=1, c="lightgray"); ax.scatter(*muestras.T, s=1, c="crimson")
    ax.set_xlim(-1.5, 2.5); ax.set_ylim(-1, 1.5)
    ax.set_title(f"{p} paso(s): precisión {precision:.0%} · cobertura {cobertura:.0%}", fontsize=9)
    print(f"{p:>3} pasos → precisión {precision:.0%} · cobertura {cobertura:.0%}")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con **1 paso** el resultado es un desastre: puntos **desperdigados por todas partes** (precisión ≈ 3 %). Desde ruido puro, la red tiene que adivinar de golpe el punto limpio, y cualquier error se amplifica.
2. La calidad sube rápido y luego se estanca: 3 pasos ≈ 45 % (ya se adivina la forma, muy emborronada), **10 pasos ≈ 83 %** y 100 pasos ≈ 88 %. Pasar de 10 a 100 pasos multiplica por 10 el costo para ganar unos pocos puntos.

En la práctica se busca el **mínimo de pasos** con calidad aceptable. Por eso existen muestreadores rápidos (como el DDIM que usamos) y modelos "destilados" que generan en 1–4 pasos.
</details>

---
## 5. 📖 Hiperparámetros y perillas

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `T` (pasos de difusión al entrenar) | En cuántos escalones se divide el camino dato → ruido | Escalones más finos; la red debe aprender más niveles | Saltos más bruscos, más difíciles de deshacer | 1000 en imágenes · 50–200 en juguetes |
| Calendario de ruido (`tipo`) | Cómo se reparte el ruido entre los pasos | — | — | Coseno (destruye la señal más gradualmente que el lineal) |
| Pasos de muestreo (`pasos`) | Cuántas veces se llama a la red al generar | Más calidad, más lento | Más rápido; con muy pocos, resultados borrosos o "promedio" | 20–50 (con DDIM u otros muestreadores rápidos) |
| Pasos de entrenamiento | Cuánto practica la red | Mejor, hasta estancarse | Muestras a medio cocinar | Mirar las muestras y la curva |
| Escala de guía (*guidance scale*, *CFG*) | Cuánto obedece a la condición (p. ej. el texto) | Más fiel al texto, menos variedad, colores exagerados | Más variedad, menos fiel | 5–8 en Stable Diffusion |
| Tamaño de la red | Capacidad (en imágenes, una **U-Net** o un Transformer) | Más detalle, más lento | Más rápido | Según el presupuesto |

### 🔮 Ejercicio 7 — El calendario de ruido
Dibujamos cuánta **señal** queda en cada paso con el calendario **lineal** y con el **coseno** (no hay que entrenar nada).
¿Cuál destruye la información **más deprisa** al principio? ¿Por qué podría ser un problema?

✍️ *Mi predicción:*

In [ ]:
for tipo in ["lineal", "coseno"]:
    s = calendario(100, tipo)
    plt.plot(range(1, 101), s, label=tipo)
    print(f"{tipo:<7} señal en el paso 25: {s[24]:.2f} · en el paso 50: {s[49]:.2f} · en el paso 75: {s[74]:.3f}")
plt.xlabel("paso t"); plt.ylabel("señal que queda"); plt.legend(); plt.grid(alpha=.3)
plt.title("Dos calendarios de ruido"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

El **lineal** destruye la señal **más deprisa**: en el paso 50 ya solo queda ≈ 0.28 de señal (contra ≈ 0.49 del coseno) y en el paso 75, ≈ 0.06 (contra ≈ 0.14).

Problema: con el lineal, muchos de los pasos finales son "casi puro ruido", así que la red dedica gran parte de su práctica a niveles donde ya no hay nada que aprender, y pocos pasos a los niveles intermedios donde se decide la forma. El coseno reparte el ruido de manera más pareja; por eso se volvió el estándar en muchos modelos.
</details>

### ✏️ Ejercicio 8 — ¿Cuánto tiene que practicar?
Entrenamos otro modelo con solo **300** pasos de entrenamiento (en vez de 4,000) y generamos con 100 pasos de muestreo. 🔮 ¿Cómo se verán las lunas? ✏️ Prueba luego con 1,000.

✍️ *Mi predicción:*

In [ ]:
PASOS_ENTRENAMIENTO = 300                      # ✏️ prueba 1000
modelo_corto = entrenar_difusion(pasos=PASOS_ENTRENAMIENTO)
muestras = generar(modelo_corto)
precision, cobertura = evaluar(muestras)
plt.figure(figsize=(5, 3))
plt.scatter(*lunas_np.T, s=1, c="lightgray"); plt.scatter(*muestras.T, s=1, c="crimson")
plt.title(f"{PASOS_ENTRENAMIENTO} pasos de entrenamiento: precisión {precision:.0%} · cobertura {cobertura:.0%}")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con 300 pasos de entrenamiento las lunas salen **a medio cocinar**: se adivina la forma, pero hay muchos puntos sueltos y las lunas se ven anchas y borrosas (precisión ≈ **51 %** contra ≈ 88 % con 4,000 pasos). La cobertura sigue siendo alta (≈ 97 %) porque los puntos están desparramados por todas partes, incluida la forma real: por eso conviene mirar **las dos métricas** a la vez.

Al contrario que en una GAN, aquí más entrenamiento casi siempre ayuda (hasta que la curva de pérdida se estanca).
</details>

---
## 6. 📖 Guiar la generación: "dibújame la luna de arriba"

Stable Diffusion no genera *cualquier* imagen: genera **la que describes con texto**. ¿Cómo? Durante el entrenamiento, la red recibe además una **descripción** del dato (en Stable Diffusion, el texto convertido en *embeddings* por un Transformer, Lección 12.4). Así aprende a quitar el ruido **"en la dirección" de esa descripción**.

En nuestro juguete, la "descripción" será simplemente **qué luna es**: 0 (la de arriba) o 1 (la de abajo).

### 🔮 Ejercicio 9 — Pide una luna
Entrenamos un modelo **guiado** y le pedimos 1,000 puntos de la luna 0 y 1,000 de la luna 1. ¿Qué porcentaje de puntos crees que caerá en la luna correcta?

✍️ *Mi predicción:*

In [ ]:
modelo_guiado = entrenar_difusion(etiquetas=ETIQUETAS)
vecino = NearestNeighbors(n_neighbors=1).fit(lunas_np)
plt.figure(figsize=(5, 3)); plt.scatter(*lunas_np.T, s=1, c="lightgray")
for luna, color in [(0, "crimson"), (1, "royalblue")]:
    muestras = generar(modelo_guiado, n=1000, etiqueta=luna)
    luna_real_mas_cercana = luna_de[vecino.kneighbors(muestras)[1][:, 0]]
    print(f"Pedí la luna {luna} → {np.mean(luna_real_mas_cercana == luna):.0%} de los puntos cayó en esa luna")
    plt.scatter(*muestras.T, s=1, c=color, label=f"pedí luna {luna}")
plt.legend(markerscale=8); plt.title("Generación guiada"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **100 %** de los puntos cae en la luna pedida, en los dos casos. Con solo un número extra en la entrada, la red aprendió a quitar el ruido "en dirección a" la luna que le pedimos.

Esto es, en miniatura, lo que hace Stable Diffusion con un texto: en lugar de "luna 0", recibe la descripción "un gato astronauta" convertida en números, y en cada paso quita el ruido en la dirección de esa descripción.
</details>

### 📖 Del juguete a Stable Diffusion
Nuestro juguete tiene todas las piezas esenciales. Los sistemas reales cambian la escala:

| Pieza | Nuestro juguete | Stable Diffusion |
|---|---|---|
| Datos | Puntos 2D | Imágenes, **comprimidas** primero por un VAE (Lección 13.2) en un "espacio latente" de 64×64×4 |
| Red que adivina el ruido | Perceptrón de 3 capas | **U-Net** (Módulo 10) o Transformer de cientos de millones de parámetros |
| Guía | "luna 0 / luna 1" | Un texto convertido en *embeddings* por un codificador de texto |
| `T` / pasos de muestreo | 100 / 1–100 | 1000 / 20–50 |
| Entrenamiento | Segundos en CPU | Semanas en cientos de GPUs, con miles de millones de imágenes |

## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Las muestras salen como una mancha "promedio" | Muy pocos pasos de muestreo | Subir los pasos o usar un muestreador mejor |
| Muestras con mucho ruido residual | Pocos pasos de entrenamiento, o la red no recibe bien el paso *t* | Entrenar más; revisar cómo se le pasa *t* a la red |
| El calendario no llega a ruido puro en el paso *T* | Al generar se empieza con ruido puro, pero la red nunca lo vio | Usar un calendario que termine con señal ≈ 0 (como el coseno) |
| La imagen no obedece al texto | Escala de guía demasiado baja | Subir la *guidance scale* |
| Colores saturados y poca variedad | Escala de guía demasiado alta | Bajarla |
| Generar es lentísimo | Es el precio de la difusión: muchas llamadas a la red | Menos pasos, modelos destilados, trabajar en espacio latente |

---
## 8. 📋 Ficha del modelo

| | Modelo de difusión |
|---|---|
| **Tipo** | Generativo |
| **Intuición** | Ver al revés la película de una gota de tinta disolviéndose: quitar ruido paso a paso |
| **Cómo aprende** | Regresión: a partir de un dato ensuciado y su nivel de ruido, **adivinar el ruido** (error cuadrático) |
| **Hiperparámetros clave** | `T`, calendario de ruido, pasos de muestreo, escala de guía |
| **✅ Fortalezas** | Calidad y **variedad** altísimas · Entrenamiento **estable** (sin juego entre redes) · Fácil de guiar con texto |
| **❌ Debilidades** | Generar es **lento** (muchas llamadas a la red) · Entrenar a gran escala es carísimo |
| **Úsalo cuando…** | Quieres generar imágenes, audio o video de alta calidad, sobre todo a partir de texto |
| **Evítalo cuando…** | Necesitas generar en tiempo real con muy poco cómputo (una GAN puede ser mejor) |

## 9. 🧠 Resumen en 5 puntos
1. La **difusión** aprende a **quitar ruido**: el proceso hacia adelante ensucia los datos con una fórmula fija; la red aprende el proceso **inverso**.
2. La red se entrena con una tarea supervisada sencilla y **estable**: adivinar el ruido que se añadió a un dato en un paso *t* al azar.
3. Para **generar**, se empieza con ruido puro y se limpia paso a paso; los primeros pasos deciden la forma y los últimos pulen detalles.
4. Perillas clave: **pasos de muestreo** (calidad vs. velocidad), **calendario de ruido**, `T` y la **escala de guía**.
5. Dándole a la red una **condición** (una etiqueta, un texto) se **guía** lo que genera: así funciona Stable Diffusion, que además trabaja en el espacio latente de un VAE.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** En un modelo de difusión, ¿qué parte se **aprende**?
- a) El proceso hacia adelante (añadir ruido) · b) El proceso inverso (quitar ruido) · c) Los dos · d) Ninguno

**2.** ¿Qué predice normalmente la red durante el entrenamiento?
- a) La etiqueta del dato · b) Si el dato es real o falso · c) El ruido que se le añadió · d) El paso siguiente del texto

**3.** Generas con 2 pasos de muestreo en lugar de 50. ¿Qué esperas?
- a) Mejor calidad · b) Más rápido pero resultados más borrosos o "promedio" · c) Lo mismo · d) Un error

**4.** Comparado con una GAN, un modelo de difusión típicamente…
- a) Es más inestable al entrenar · b) Es más estable al entrenar pero más lento al generar · c) Genera en un solo paso · d) No necesita datos

**5.** En Stable Diffusion subes mucho la escala de guía (*guidance scale*). ¿Qué esperas?
- a) Imágenes más fieles al texto pero menos variadas · b) Imágenes al azar · c) Más variedad · d) Nada

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El proceso hacia adelante es una fórmula fija.
2. **c)** Adivinar el ruido (y con eso, restárselo).
3. **b)** Es lo que viste en el Ejercicio 6.
4. **b)** Regresión estable, pero muchas llamadas a la red al generar.
5. **a)** Obedece más al texto, a costa de la variedad.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué Stable Diffusion aplica la difusión sobre el **espacio latente de un VAE** (una versión comprimida de la imagen) en lugar de hacerlo directamente sobre los píxeles?

**B. Caso de análisis 💼** Una app de filtros para fotos quiere generar un fondo nuevo **en tiempo real** (30 veces por segundo) en teléfonos. Un equipo propone un modelo de difusión con 50 pasos; otro, una GAN. ¿Qué preguntarías y qué recomendarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Por **costo**: una imagen de 512×512×3 tiene ≈ 786,000 números; su versión latente, 64×64×4 ≈ 16,000, unas **48 veces menos**. Cada uno de los muchos pasos de difusión se vuelve mucho más barato. El VAE se encarga de los detalles finos al decodificar, y la difusión de la "idea" de la imagen.

**B.** 50 pasos × 30 imágenes por segundo = 1,500 llamadas a una red grande por segundo en un teléfono: inviable hoy. Preguntas: ¿cuánta variedad y calidad se necesita?, ¿cuánta latencia se tolera?, ¿existen versiones **destiladas** de difusión de 1–4 pasos? Recomendación razonable: una **GAN** (o un modelo de difusión destilado a muy pocos pasos) para el tiempo real, y difusión completa solo para funciones "premium" donde el usuario espera unos segundos.
</details>

### 10.3 🧩 Práctica integradora: "Difusión para una forma nueva"

Ahora entrenarás un modelo de difusión para una forma **distinta**: una **espiral**. Tu objetivo: encontrar el **mínimo número de pasos de muestreo** que da una precisión de al menos **80 %** (menos pasos = generación más barata).

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos.**

In [ ]:
rng_esp = np.random.default_rng(3)
giro = np.sqrt(rng_esp.uniform(0, 1, 3000)) * 3 * np.pi                 # ángulo a lo largo de la espiral
espiral_np = np.c_[giro * np.cos(giro), giro * np.sin(giro)] / 5 + rng_esp.normal(0, 0.03, (3000, 2))
ESPIRAL = torch.tensor(espiral_np, dtype=torch.float32)
plt.figure(figsize=(4, 4)); plt.scatter(*espiral_np.T, s=1); plt.title("Datos reales: espiral"); plt.axis("equal"); plt.show()

**Paso 2 🔮 — ¿Más fácil o más difícil que las lunas?** Con el mismo modelo pequeño y los mismos 4,000 pasos de entrenamiento, ¿esperas mejor o peor precisión que con las lunas? ¿Por qué?

✍️ *Mi apuesta:*

**Paso 3 ▶️ — Entrena y genera con 100 pasos.**

In [ ]:
modelo_esp = entrenar_difusion(datos=ESPIRAL)
muestras = generar(modelo_esp)
precision, cobertura = evaluar(muestras, reales=espiral_np)
plt.figure(figsize=(4, 4)); plt.scatter(*espiral_np.T, s=1, c="lightgray"); plt.scatter(*muestras.T, s=1, c="crimson")
plt.title(f"100 pasos: precisión {precision:.0%} · cobertura {cobertura:.0%}"); plt.axis("equal"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Lo razonable era esperar algo **más difícil**: la espiral tiene curvas más cerradas y una parte central estrecha. Sorpresa: con 100 pasos de muestreo la precisión es ≈ **91 %** y la cobertura ≈ **100 %**, incluso algo mejor que con las lunas (≈ 88 %). En la figura se ve que la parte exterior sale muy bien y la **parte central**, donde la espiral se enrosca, es la más desordenada.

Lección: la dificultad de una forma no siempre es la que intuimos; por eso se **mide**.
</details>

**Paso 4 ✏️ — Busca el mínimo número de pasos.** Recorre varios valores de pasos de muestreo y encuentra el menor que llega a 80 % de precisión.

In [ ]:
for p in [2, 5, 10, 20, 50, 100]:              # ✏️ ajusta la lista
    precision, cobertura = evaluar(generar(modelo_esp, pasos=p), reales=espiral_np)
    print(f"{p:>3} pasos → precisión {precision:.0%} · cobertura {cobertura:.0%}" + ("  ✅" if precision >= 0.8 else ""))

✍️ *Mi número mínimo de pasos:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con 2 pasos la precisión es casi nula (≈ 2 %), pero con **5 pasos** ya supera el 80 % (≈ 91 %). A partir de ahí la precisión se mueve alrededor del 90 % (sube y baja uno o dos puntos por azar), y lo que sigue mejorando un poco es la cobertura (≈ 95 % → 100 %).

Mínimo: **5 pasos**. Si también quieres una cobertura de casi 100 %, ≈ 20 pasos.
</details>

**Paso 5 ✋ — Informe de una línea.** Si cada llamada a la red tardara 20 ms en el servidor, ¿cuánto tardaría generar una muestra con 100 pasos y con tu mínimo? ¿Qué le recomendarías al equipo de producto?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con 100 pasos: 100 × 20 ms = **2 s** por muestra. Con 5 pasos: 5 × 20 ms = **0.1 s**, unas **20 veces más rápido** con una calidad prácticamente igual en precisión.

Informe: *"Recomiendo generar con 5–10 pasos: la calidad es casi la misma que con 100 y la respuesta pasa de 2 s a 0.1–0.2 s. Si aparecen formas con huecos, subir a 20 pasos."* La regla general: **el número de pasos de muestreo es una perilla de costo, y se ajusta midiendo**.
</details>

---
## 📚 Para profundizar
- 📘 **Artículos clave:** Ho et al. (2020), *Denoising Diffusion Probabilistic Models* (DDPM); Song et al. (2020), *DDIM*; Rombach et al. (2022), *High-Resolution Image Synthesis with Latent Diffusion Models* (Stable Diffusion).
- 📺 **Video:** Computerphile, "How AI Image Generators Work (Stable Diffusion / Dall-E)" (inglés, con subtítulos).
- 📝 **Blog:** Lilian Weng, *What are Diffusion Models?* (más matemático, para quien quiera ir más allá).
- 🛠️ **Para Colab (con GPU):** la librería `diffusers` de Hugging Face permite generar imágenes con Stable Diffusion en pocas líneas:
  ```python
  # Solo en Colab con GPU; descarga varios GB
  from diffusers import StableDiffusionPipeline
  pipe = StableDiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-2-1-base").to("cuda")
  pipe("un gato astronauta, acuarela", num_inference_steps=25, guidance_scale=7.5).images[0]
  ```

---
⬅️ **Anterior:** [13.3 GANs: el falsificador y el detective](13.3-gans.ipynb) · ➡️ **Siguiente:** [13.5 Cómo se entrena un LLM: pre-entrenamiento, ajuste por instrucciones, RLHF/RLAIF](13.5-como-se-entrena-un-llm.ipynb)